<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>06. 🔣 Custom Regex Interpreter from Scratch</b>
</div>


### 📌 Project Overview
In this project, we implement a custom regular expression matching engine from scratch, entirely bypassing the built-in `re` module.
We'll explore parsing mechanics, state match trackers, and construct a recursive backtracking search engine that supports standard pattern matching syntaxes.

#### 🔑 Key Concepts Covered:
- Recursive backtracking algorithms for state resolution
- Simulating automata without pre-built regex parsers
- Mapping patterns matching operators like wildcards `.`, anchors `^`/`$`, and multi-char repeats `*`/`+`/`?`
- Analyzing pattern search complexities and matching states


In [1]:
class CustomRegexEngine:
    """A simplified regular expression backtracking engine."""
    
    def match_anchored(self, pattern, text):
        """Evaluate if the pattern matches at the current index of text."""
        # If pattern is empty, matching succeeded
        if len(pattern) == 0:
            return True
            
        # Handle end-of-text anchor '$'
        if pattern == '$':
            return len(text) == 0
            
        # Handle operators with repeat counts (*, +, ?)
        if len(pattern) > 1 and pattern[1] in ('*', '+', '?'):
            op = pattern[1]
            char = pattern[0]
            sub_pattern = pattern[2:]
            
            if op == '*':
                # Match 0 or more occurrences
                i = 0
                while True:
                    if self.match_anchored(sub_pattern, text[i:]):
                        return True
                    if i < len(text) and (text[i] == char or char == '.'):
                        i += 1
                    else:
                        break
                return False
                
            elif op == '+':
                # Match 1 or more occurrences: first char must match, then fallback to *
                if len(text) > 0 and (text[0] == char or char == '.'):
                    return self.match_anchored(char + '*' + sub_pattern, text[1:])
                return False
                
            elif op == '?':
                # Match 0 or 1 occurrence
                if len(text) > 0 and (text[0] == char or char == '.'):
                    if self.match_anchored(sub_pattern, text[1:]):
                        return True
                return self.match_anchored(sub_pattern, text)
                
        # Handle single character or wildcard matching
        if len(text) > 0 and (pattern[0] == text[0] or pattern[0] == '.'):
            return self.match_anchored(pattern[1:], text[1:])
            
        return False
        
    def match(self, pattern, text):
        # If pattern begins with '^', anchor search to the start of text
        if pattern.startswith('^'):
            return self.match_anchored(pattern[1:], text)
            
        # Match anywhere in the text
        idx = 0
        while idx <= len(text):
            if self.match_anchored(pattern, text[idx:]):
                return True
            idx += 1
        return False

In [2]:
engine = CustomRegexEngine()

tests = [
    # (pattern, search_text, expected_outcome)
    ('^hello', 'hello world', True),
    ('^hello', 'say hello', False),
    ('py.hon', 'python coding', True),
    ('py.hon', 'jython compiler', False),
    ('ab*c', 'ac', True),           # 0 occurrences of b
    ('ab*c', 'abbbbc', True),       # Multiple occurrences of b
    ('ab+c', 'ac', False),          # 0 occurrences of b fails
    ('ab+c', 'abc', True),
    ('colou?r', 'color', True),     # Optional u
    ('colou?r', 'colour', True),    # Optional u
    ('finish$', 'will finish', True),
    ('finish$', 'finish line', False)
]

print('='*50)
print(f'{"Pattern":<12} | {"Search Text":<16} | {"Result":<6}')
print('='*50)
for pat, text, expected in tests:
    res = engine.match(pat, text)
    status = '✅ PASS' if res == expected else '❌ FAIL'
    print(f'{pat:<12} | {text:<16} | {str(res):<6} ({status})')
print('='*50)


Pattern      | Search Text      | Result
^hello       | hello world      | True   (✅ PASS)
^hello       | say hello        | False  (✅ PASS)
py.hon       | python coding    | True   (✅ PASS)
py.hon       | jython compiler  | False  (✅ PASS)
ab*c         | ac               | True   (✅ PASS)
ab*c         | abbbbc           | True   (✅ PASS)
ab+c         | ac               | False  (✅ PASS)
ab+c         | abc              | True   (✅ PASS)
colou?r      | color            | True   (✅ PASS)
colou?r      | colour           | True   (✅ PASS)
finish$      | will finish      | True   (✅ PASS)
finish$      | finish line      | False  (✅ PASS)
